# Histopathology Patch Metadata — Overview
Loads `pathoImagesWithLabels.xlsx`, counts unique patients/polyps per type, and plots the polyp-type distribution across the histopathology image collection.

**Requires:** `data/pathoImagesWithLabels.xlsx`.

In [ ]:
# --- VIM-Polyp setup: resolve DATA_DIR / OUTPUT_DIR (works on Kaggle, Colab, and a local clone) ---
import subprocess
import sys
from pathlib import Path


def _find_or_fetch_src_dir():
    for base in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        candidate = base / "src"
        if (candidate / "vimpolyp_paths.py").exists():
            return candidate
    # Not inside a checkout of the repo (e.g. a bare Colab session) -> clone it.
    repo_dir = Path("VIM-Polyp")
    if not (repo_dir / "src" / "vimpolyp_paths.py").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/biorsd/VIM-Polyp.git", str(repo_dir)],
            check=True,
        )
    return repo_dir / "src"


sys.path.insert(0, str(_find_or_fetch_src_dir()))
from vimpolyp_paths import DATA_DIR, OUTPUT_DIR  # noqa: E402

print(f"Data directory:   {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

# If DATA_DIR is empty, fetch the dataset first, e.g. from the repo root:
#   python scripts/download_zenodo.py --dest data/
#   python scripts/download_kaggle.py --dest data/

In [ ]:
import matplotlib.pyplot as plt

subfolders = ['2,5x', '5x', '10x', '20x', '40x']
image_counts = [393, 397, 400, 399, 316]

plt.figure(figsize=(10, 6))
plt.bar(subfolders, image_counts)
plt.xlabel('Zoom Levels')
plt.ylabel('Number of Images')
plt.title('Number of Images by Zoom Levels')
plt.xticks(rotation=45)
plt.show()

This code reads the Excel file 'pathoImagesWithLabels.xlsx' and extracts the necessary columns. It then calculates the number of unique patients (unique_patient_count) and the number of unique polyps (unique_polyp_count). Finally, it calculates the number of unique polyps per patient (polyp_count_per_patient) and the total number of unique polyps (total_unique_polyp_count).

In [ ]:
import pandas as pd

df = pd.read_excel(str(DATA_DIR / "pathoImagesWithLabels.xlsx"), sheet_name='Sheet1')

# Find the number of unique patients
unique_patient_count = df["patient"].nunique()

# Find the number of unique polyps
unique_polyp_count = df["polyp"].nunique()

# Print the results
print("Total Unique Patient Count:", unique_patient_count)
print("Total Unique Polyp Count:", unique_polyp_count)


# Calculate the number of unique polyps per patient
polyp_count_per_patient = df.groupby("patient")["polyp"].nunique()

# Calculate the total number of unique polyps
total_unique_polyp_count = polyp_count_per_patient.sum()

# Print the results
print("Total Unique Polyp Count (Per Patient):", total_unique_polyp_count)


In [ ]:
import seaborn as sns
# Bar plot showing the count of polyp types
plt.figure(figsize=(10,6))
sns.countplot(data=df, x='type')
plt.title('Polyp Type Counts')


output_path = str(OUTPUT_DIR / "polyp_type_counts.png") 
plt.savefig(output_path, bbox_inches='tight', dpi=300) # bbox_inches='tight' kenar boşluklarını ayarlar, dpi çözünürlüğü artırır.
plt.show()

df.loc[df['type'] == 'carcinom', 'type'] = 'carcinoma'
# Calculate the counts of polyp types
polyp_type_counts = df['type'].value_counts()

# Print the results
print(polyp_type_counts)

In [ ]:
df.loc[df['subtype'] == 'serrated', 'subtype'] = 'hyperplastic'
df.loc[df['subtype'] == 'inflammation', 'subtype'] = 'hyperplastic'
df.loc[df['subtype'] == 'nonspesific', 'subtype'] = 'hyperplastic'

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get the distribution of values in the 'subtype' column
subtype_distribution = df['subtype'].value_counts()
df.loc[df['subtype'] == 'carcinom', 'subtype'] = 'carc'
df.loc[df['subtype'] == 'inflamatuar', 'subtype'] = 'inf'
df.loc[df['subtype'] == 'hyperplastic', 'subtype'] = 'hyper'
df.loc[df['subtype'] == 'tubulovillous', 'subtype'] = 'tvillous'
df.loc[df['subtype'] == 'nonspesific', 'subtype'] = 'hyper'
df.loc[df['subtype'] == 'serrated', 'subtype'] = 'hyper'
df.loc[df['subtype'] == 'tubuvillous', 'subtype'] = 'tvillous'



# Get the names and counts of the subtypes as separate lists
subtypes = subtype_distribution.index.tolist()
counts = subtype_distribution.values.tolist()

# Create a bar plot
plt.figure(figsize=(10,6))
sns.barplot(x=subtypes, y=counts, alpha=0.8)

plt.title('Distribution of Subtypes')
plt.ylabel('Number of Occurrences', fontsize=12)
plt.xlabel('Subtype', fontsize=12)
output_path = str(OUTPUT_DIR / "polyp_type_counts.png") 
plt.savefig(output_path, bbox_inches='tight', dpi=300) # bbox_inches='tight' kenar boşluklarını ayarlar, dpi çözünürlüğü artırır.

plt.show()


import pandas as pd

# Perform grouping to count polyp subtypes
polyp_subtype_counts = df.groupby('subtype').size().reset_index(name='count')

# Print the results using DataFrame
print(polyp_subtype_counts)

In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

# Read the data
pathology_reports = pd.read_excel(str(DATA_DIR / "pathoImagesWithLabels.xlsx"), sheet_name='forKappa')
ihc_reports = pd.read_excel(str(DATA_DIR / "pathoImagesWithLabels.xlsx"), sheet_name='forKappa')

# Extract the 'subtype' columns
pathology_subtype = pathology_reports['subtypePathology']
ihc_subtype = ihc_reports['subtypeIHK']

# First, create a DataFrame that pairs the 'subtype' columns from both DataFrames
paired_subtypes = pd.DataFrame({
    'pathology_subtype': pathology_subtype,
    'ihc_subtype': ihc_subtype
})

# Drop rows where either value is NaN
paired_subtypes = paired_subtypes.dropna()

# Now you can calculate Cohen's Kappa on the remaining pairs
kappa = cohen_kappa_score(paired_subtypes['pathology_subtype'], paired_subtypes['ihc_subtype'])

print("Cohen's Kappa:", kappa)
